# KWGT Miner - Optimized with Local Ingest

This notebook mines KWGT files for internal_type registry, extracts preset.json, and analyzes KBM structures.

## Features
- Local ingest from Google Drive (MyDrive/kwgt_local_input)
- Extract preset.json only from .kwgt files
- Mine internal_type registry
- Generate module/schema analysis
- Export results as ZIP

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q zipfile36

In [ ]:
import os
import json
import zipfile
from pathlib import Path
from collections import defaultdict, Counter
import shutil

# Configuration
INPUT_DIR = Path('/content/drive/MyDrive/kwgt_local_input')
OUTPUT_DIR = Path('/content/kwgt_output')
EXTRACTED_DIR = OUTPUT_DIR / 'extracted_presets'
REGISTRY_FILE = OUTPUT_DIR / 'internal_type_registry.json'
ANALYSIS_FILE = OUTPUT_DIR / 'schema_analysis.json'

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
EXTRACTED_DIR.mkdir(exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def extract_preset_from_kwgt(kwgt_path, output_dir):
    """
    Extract preset.json from a .kwgt file.
    
    Args:
        kwgt_path: Path to .kwgt file
        output_dir: Directory to save extracted preset.json
    
    Returns:
        Path to extracted preset.json or None if failed
    """
    try:
        with zipfile.ZipFile(kwgt_path, 'r') as zip_ref:
            # Check if preset.json exists
            if 'preset.json' not in zip_ref.namelist():
                print(f"⚠️  No preset.json in {kwgt_path.name}")
                return None
            
            # Extract preset.json
            preset_data = zip_ref.read('preset.json')
            
            # Save with original filename
            output_path = output_dir / f"{kwgt_path.stem}_preset.json"
            with open(output_path, 'wb') as f:
                f.write(preset_data)
            
            return output_path
    except Exception as e:
        print(f"❌ Error extracting {kwgt_path.name}: {e}")
        return None

def collect_internal_types(obj, types_dict=None, path="root"):
    """
    Recursively collect all internal_type values from KBM structure.
    
    Args:
        obj: KBM object or structure
        types_dict: Dictionary to store types and their contexts
        path: Current path in the structure
    
    Returns:
        Dictionary of internal_type -> list of contexts
    """
    if types_dict is None:
        types_dict = defaultdict(list)
    
    if isinstance(obj, dict):
        # Check for internal_type
        if 'internal_type' in obj:
            internal_type = obj['internal_type']
            types_dict[internal_type].append({
                'path': path,
                'keys': list(obj.keys())[:20]  # Limit keys to avoid huge output
            })
        
        # Recurse into nested objects
        for key, value in obj.items():
            collect_internal_types(value, types_dict, f"{path}.{key}")
    
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            collect_internal_types(item, types_dict, f"{path}[{i}]")
    
    return types_dict

def analyze_schema(preset_data):
    """
    Analyze KBM schema structure.
    
    Returns:
        Dictionary with schema analysis
    """
    analysis = {
        'has_root_layer': 'root_layer' in preset_data,
        'has_globals': 'globals' in preset_data,
        'has_items': 'items' in preset_data,
        'globals_count': len(preset_data.get('globals', [])),
        'items_count': len(preset_data.get('items', [])),
        'root_type': preset_data.get('root_layer', {}).get('internal_type', 'N/A'),
    }
    
    return analysis

In [ ]:
# Process all .kwgt files in input directory
kwgt_files = list(INPUT_DIR.glob('*.kwgt')) if INPUT_DIR.exists() else []

print(f"Found {len(kwgt_files)} .kwgt files")
print("="*60)

# Statistics
extracted_count = 0
failed_count = 0
all_internal_types = defaultdict(list)
schema_analyses = []

# Process each file
for kwgt_file in kwgt_files:
    print(f"\nProcessing: {kwgt_file.name}")
    
    # Extract preset.json
    preset_path = extract_preset_from_kwgt(kwgt_file, EXTRACTED_DIR)
    
    if preset_path:
        extracted_count += 1
        
        try:
            # Load and analyze preset
            with open(preset_path, 'r', encoding='utf-8') as f:
                preset_data = json.load(f)
            
            # Collect internal types
            types_dict = collect_internal_types(preset_data)
            for itype, contexts in types_dict.items():
                all_internal_types[itype].extend(contexts)
            
            # Analyze schema
            analysis = analyze_schema(preset_data)
            analysis['filename'] = kwgt_file.name
            schema_analyses.append(analysis)
            
            print(f"  ✅ Extracted and analyzed")
            print(f"     Types found: {len(types_dict)}")
            print(f"     Root type: {analysis['root_type']}")
            print(f"     Globals: {analysis['globals_count']}, Items: {analysis['items_count']}")
        
        except Exception as e:
            print(f"  ⚠️  Error analyzing: {e}")
            failed_count += 1
    else:
        failed_count += 1

print("\n" + "="*60)
print(f"\n📊 Summary:")
print(f"   Total files: {len(kwgt_files)}")
print(f"   Extracted: {extracted_count}")
print(f"   Failed: {failed_count}")
print(f"   Unique internal_types: {len(all_internal_types)}")

In [ ]:
# Generate internal_type registry
registry = {}
for itype, contexts in all_internal_types.items():
    # Get unique keys across all contexts
    all_keys = set()
    for ctx in contexts:
        all_keys.update(ctx['keys'])
    
    registry[itype] = {
        'count': len(contexts),
        'common_keys': sorted(list(all_keys)),
        'example_paths': list(set(ctx['path'] for ctx in contexts[:5]))  # First 5 unique paths
    }

# Save registry
with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print(f"\n📋 Internal Type Registry:")
print(f"   Saved to: {REGISTRY_FILE}")
print(f"   Types: {len(registry)}")
print("\nTop 10 most common types:")
sorted_types = sorted(registry.items(), key=lambda x: x[1]['count'], reverse=True)[:10]
for itype, data in sorted_types:
    print(f"   - {itype}: {data['count']} occurrences")

In [ ]:
# Save schema analysis
with open(ANALYSIS_FILE, 'w', encoding='utf-8') as f:
    json.dump(schema_analyses, f, indent=2)

print(f"\n📊 Schema Analysis:")
print(f"   Saved to: {ANALYSIS_FILE}")
print(f"   Files analyzed: {len(schema_analyses)}")

# Summary statistics
if schema_analyses:
    root_types = Counter(a['root_type'] for a in schema_analyses)
    avg_globals = sum(a['globals_count'] for a in schema_analyses) / len(schema_analyses)
    avg_items = sum(a['items_count'] for a in schema_analyses) / len(schema_analyses)
    
    print(f"\nStatistics:")
    print(f"   Root types: {dict(root_types)}")
    print(f"   Avg globals per widget: {avg_globals:.1f}")
    print(f"   Avg items per widget: {avg_items:.1f}")

In [ ]:
# Create ZIP archive of all outputs
zip_output = Path('/content') / 'kwgt_mining_results.zip'

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add registry and analysis files
    zipf.write(REGISTRY_FILE, REGISTRY_FILE.name)
    zipf.write(ANALYSIS_FILE, ANALYSIS_FILE.name)
    
    # Add all extracted presets
    for preset_file in EXTRACTED_DIR.glob('*.json'):
        zipf.write(preset_file, f"extracted_presets/{preset_file.name}")

print(f"\n📦 Results packaged:")
print(f"   ZIP file: {zip_output}")
print(f"   Size: {zip_output.stat().st_size / 1024:.1f} KB")
print(f"\n✅ Mining complete! Download the ZIP file from the Files panel.")

## Usage Instructions

1. Upload .kwgt files to `MyDrive/kwgt_local_input` in your Google Drive
2. Run all cells in order
3. Download the `kwgt_mining_results.zip` file from the Files panel

## Output Structure

```
kwgt_mining_results.zip
├── internal_type_registry.json  # Registry of all internal_type values
├── schema_analysis.json         # Schema analysis for each widget
└── extracted_presets/           # All extracted preset.json files
    ├── widget1_preset.json
    ├── widget2_preset.json
    └── ...
```